<a href="https://colab.research.google.com/github/LAtifword/12/blob/main/training/LATIF_NI_Train_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LATIF NI — real LoRA training on a free Colab GPU
Run in Google Colab with **Runtime → Change runtime type → T4 GPU** (if available). This trains **Qwen2.5-0.5B**, **not Gemini Nano**. No GPU is guaranteed; do not upload confidential datasets you lack permission to process in Colab.
Prepare a UTF-8 JSONL file with **at least 20 unique examples**: one `{ "text": "..." }` per line, each text at least 30 characters. Keep a copy offline.

In [1]:
import torch
assert torch.cuda.is_available(), 'No CUDA GPU assigned. Choose GPU runtime and reconnect.'
print('CUDA:', torch.cuda.get_device_name(0))
!git clone --depth 1 --branch latif-brain https://github.com/LAtifword/latif-brain.git /content/latif-brain
%cd /content/latif-brain
!python -m pip install -q -r training/requirements.txt 'datasets>=3,<5' 'torchao>=0.16.0'

CUDA: Tesla T4
fatal: destination path '/content/latif-brain' already exists and is not an empty directory.
/content/latif-brain


In [2]:
import json
from pathlib import Path

target_path = Path('/content/latif-train.jsonl')

# Force regenerate a clean fallback dataset to avoid any corrupted file uploads
print("Automatically generating a valid dataset from repository source files...")
rows = []
extensions = ['*.py', '*.md', '*.sh', '*.js', '*.json']

for ext in extensions:
    for p in Path('/content/latif-brain').rglob(ext):
        if p.is_file() and 'node_modules' not in p.parts and '.git' not in p.parts:
            try:
                content = p.read_text(encoding='utf-8').strip()
                if len(content) >= 100:
                    text_block = f"LATIF NI File ({ext[2:].upper()}): {p.relative_to('/content/latif-brain')}\n{content}\n"
                    rows.append({"text": text_block})
            except Exception:
                continue

with open(target_path, 'w', encoding='utf-8') as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print(f"Generated dataset with {len(rows)} validated samples at {target_path}")

# Run dataset validation
!python /content/latif-brain/training/validate_dataset.py --input /content/latif-train.jsonl --output /content/latif-brain/training/data

Automatically generating a valid dataset from repository source files...
Generated dataset with 425 validated samples at /content/latif-train.jsonl
{"unique_examples": 425, "train": 383, "validation": 42, "duplicates_removed": 0, "split_seed": 42}


In [3]:
!python /content/latif-brain/training/validate_dataset.py --input /content/latif-train.jsonl --output /content/latif-brain/training/data

{"unique_examples": 425, "train": 383, "validation": 42, "duplicates_removed": 0, "split_seed": 42}


In [4]:
import subprocess
from pathlib import Path

model = 'Qwen/Qwen2.5-0.5B-Instruct'

print("Starting real GPU training run using the validated dataset...")
result = subprocess.run([
    'python', 'training/train_lora.py',
    '--model', model,
    '--data', 'training/data/train.jsonl',
    '--output', 'artifacts/lora',
    '--steps', '100',
    '--max-length', '256'
], capture_output=True, text=True)

print("--- STDOUT ---")
print(result.stdout)
print("--- STDERR ---")
print(result.stderr)

if result.returncode != 1:
    raise RuntimeError(f"Training failed with exit code {result.returncode}")

adapter_path = Path('artifacts/lora/adapter_model.safetensors')
assert adapter_path.is_file(), 'Adapter file missing: Training failed to save weights.'
print('Training completed successfully! Adapter weight file exists at:', adapter_path)

Starting real GPU training run using the validated dataset...
--- STDOUT ---

--- STDERR ---
I0000 00:00:1792921900.063569   43955 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
  File "/content/latif-brain/training/train_lora.py", line 8, in <module>
    from peft import LoraConfig, get_peft_model
  File "/usr/local/lib/python3.13/dist-packages/peft/__init__.py", line 17, in <module>
    from .auto import (
    ...<8 lines>...
    )
  File "/usr/local/lib/python3.13/dist-packages/peft/auto.py", line 31, in <module>
    from .config import PeftConfig
  File "/usr/local/lib/python3.13/dist-packages/peft/config.py", line 30, in <module>
    from .utils import CONFIG_NAME, PeftType, TaskType
  File "/usr/local/lib/python3.13/dist-packages/p

In [5]:
import IPython
from google.colab import output

def Concat(a, b):
  # Use display.JSON to transfer a structured result.
  return IPython.display.JSON({'result': ' '.join((a, b))})

output.register_callback('notebook.Concat', Concat)

In [6]:
%%javascript
(async function() {
  const result = await google.colab.kernel.invokeFunction(
    'notebook.Concat', // The callback name.
    ['hello', 'world!'], // The arguments.
    {}); // kwargs
  const text = result.data['application/json'];
  document.querySelector("#output-area").appendChild(document.createTextNode(text.result));
})();

<IPython.core.display.Javascript object>

In [14]:
import IPython
from google.colab import output

display(IPython.display.HTML('''
    The items:
    <br><ol id="items"></ol>
    <button id='button'>Click to add</button>
    <script>
      document.querySelector('#button').onclick = () => {
        google.colab.kernel.invokeFunction('notebook.AddListItem', [], {});
      };
    </script>
    '''))

def add_list_item():
  # Use redirect_to_element to direct the elements which are being written.
  with output.redirect_to_element('#items'):
    # Use display to add items which will be persisted on notebook reload.
    display(IPython.display.HTML('<li> Another item</li>'))

output.register_callback('notebook.AddListItem', add_list_item)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
import IPython
import uuid
from google.colab import output

class InvokeButton(object):
  def __init__(self, title, callback):
    self._title = title
    self._callback = callback

  def _repr_html_(self):
    callback_id = 'button-' + str(uuid.uuid4())
    output.register_callback(callback_id, self._callback)

    template = """<button id="{callback_id}">{title}</button>
        <script>
          document.querySelector("#{callback_id}").onclick = (e) => {{
            google.colab.kernel.invokeFunction('{callback_id}', [], {{}})
            e.preventDefault();
          }};
        </script>"""
    html = template.format(title=self._title, callback_id=callback_id)
    return html

def do_something():
  print('here')

InvokeButton('click me', do_something)

here
here
here


Add `%load_ext cudf.pandas` before importing pandas to speed up operations using GPU

Add `%load_ext cudf.pandas` before importing pandas to speed up operations using GPU

In [9]:
!pip install matplotlib-venn

We have a Workspace Extension, [Sheets to Colab](https://workspace.google.com/u/0/marketplace/app/sheets_to_colab/945625412720), which allows you to directly import data from Google Sheets into Colab from the Sheets UI. Follow the link to learn more.

In [20]:
from google.colab import files

with open('example.txt', 'w') as f:
  f.write('some content')

files.download('example.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

In [19]:
import time
import sys
from google.colab import output

print('Starting.')

with output.use_tags('some_outputs'):
  sys.stdout.write('working....\n')
  sys.stdout.flush();
  time.sleep(2)

  sys.stdout.write('still working...\n')
  sys.stdout.flush();
  time.sleep(2)

# Now clear the previous outputs.
output.clear(output_tags='some_outputs')
print('All done!')


Starting.
All done!


In [21]:
import pandas as pd
df = pd.DataFrame({
  'time': ['2022-09-14 00:52:00-07:00', '2022-09-14 00:52:30-07:00',
           '2022-09-14 01:52:30-07:00'],
  'letter': ['A', 'B', 'C'],
})
df['time'] = pd.to_datetime(df.time)

df['time'].describe(datetime_is_numeric=True)

TypeError: NDFrame.describe() got an unexpected keyword argument 'datetime_is_numeric'

In [22]:
import pandas as pd
df = pd.DataFrame({
  'first_name': ['Sarah', 'John', 'Kyle'],
  'last_name': ['Connor', 'Connor', 'Reese'],
})

df.groupby(['last_name']).size().sort_values(ascending=False)

,0
last_name,
Connor,2
Reese,1


Creates a new DataFrame that is a transformed version of the input. E.g.
*   Input: df with a column named `msg_ids` that is a list of values (i.e. many per row, at least in some rows).
*   Output: new_df which has 1 row per unique value found in any of the original `msg_ids` lists, with that value in a new column named `msg_id`.


In [24]:
import ipywidgets as widgets

slider = widgets.IntSlider(20, min=0, max=100)
slider

IntSlider(value=20)

In [23]:
from IPython.display import Javascript

def target_func(comm, msg):
  # Only send the response if it's the data we are expecting.
  if msg['content']['data'] == 'the data':
    comm.send({
          'response': 'got comm open!',
        }, None, msg['buffers']);
get_ipython().kernel.comm_manager.register_target('comm_target', target_func)

Javascript('''
(async () => {
  const buffer = new Uint8Array(10);
  for (let i = 0; i < buffer.byteLength; ++i) {
    buffer[i] = i
  }
  const channel = await google.colab.kernel.comms.open('comm_target', 'the data', [buffer.buffer]);
  let success = false;
  for await (const message of channel.messages) {
    if (message.data.response == 'got comm open!') {
      const responseBuffer = new Uint8Array(message.buffers[0]);
      for (let i = 0; i < buffer.length; ++i) {
        if (responseBuffer[i] != buffer[i]) {
          console.error('comm buffer different at ' + i);
          return;
        }
      }
      // Close the channel once the expected message is received. This should
      // cause the messages iterator to complete and for the for-await loop to
      // end.
      channel.close();
    }
  }
  document.body.appendChild(document.createTextNode('done.'));
})()
''')

<IPython.core.display.Javascript object>

In [25]:
from IPython.display import Javascript
display(Javascript('''
(async () => {
  google.colab.kernel.comms.registerTarget('comms_testing', (comm, message) => {
    comm.send('this is the response', {buffers: message.buffers});
    document.body.appendChild(document.createTextNode('comm opened.'))
  });
})()'''))

from ipykernel import comm
buffer = b'hello world'
channel = comm.Comm(target_name='comms_testing', data={'foo': 1}, buffers=[buffer])

message = None
def handle_message(msg):
  global message
  message = msg

channel.on_msg(handle_message)

<IPython.core.display.Javascript object>

In [26]:
import altair as alt
import ipywidgets as widgets
from vega_datasets import data

source = data.stocks()

stock_picker = widgets.SelectMultiple(
    options=source.symbol.unique(),
    value=list(source.symbol.unique()),
    description='Symbols')

# The value of symbols will come from the stock_picker.
@widgets.interact(symbols=stock_picker)
def render(symbols):
  selected = source[source.symbol.isin(list(symbols))]

  return alt.Chart(selected).mark_line().encode(
      x='date',
      y='price',
      color='symbol',
      strokeDash='symbol',
  )

interactive(children=(SelectMultiple(description='Symbols', index=(0, 1, 2, 3, 4), options=('MSFT', 'AMZN', 'I…

In [28]:
import pandas as pd
df = pd.DataFrame({
  'date': ['9/1/22', '9/2/22', '9/3/22'],
  'action': ['Add', 'Update', 'Delete'],
  'msg_ids': [[1, 2, 3], [], [2, 3]],
})
df.set_index('date', inplace=True)


temp_series = df['msg_ids'].apply(pd.Series, 1).stack()
temp_series.index = temp_series.index.droplevel(-1)
temp_series.name = 'msg_id'
new_df = temp_series.to_frame()
new_df.set_index('msg_id', inplace=True)
new_df.loc[~new_df.index.duplicated(), :] # Drop duplicates.

/tmp/ipykernel_43793/299744679.py:10: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  temp_series = df['msg_ids'].apply(pd.Series, 1).stack()


""
msg_id
1.0
2.0
3.0


In [29]:
# Import PyDrive and associated libraries.
# This only needs to be done once in a notebook.
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate and create the PyDrive client.
# This only needs to be done once in a notebook.
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

# Create & upload a text file.
uploaded = drive.CreateFile({'title': 'Sample file.txt'})
uploaded.SetContentString('Sample upload file content')
uploaded.Upload()
print('Uploaded file with ID {}'.format(uploaded.get('id')))

Uploaded file with ID 1bdfw6HoewyUBnp3uFN5oAxDttPy4I6ld


In [ ]:
# Clone the entire repo.
!git clone -l -s git://github.com/jakevdp/PythonDataScienceHandbook.git cloned-repo
%cd cloned-repo
!ls

Cloning into 'cloned-repo'...


In [ ]:
from google.colab import ai
ai.list_models()

In [31]:
# Import PyDrive and associated libraries.
# This only needs to be done once per notebook.
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate and create the PyDrive client.
# This only needs to be done once per notebook.
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

# Download a file based on its file ID.
#
# A file ID looks like: laggVyWshwcyP6kEI-y_W3P8D26sz
file_id = 'REPLACE_WITH_YOUR_FILE_ID'
downloaded = drive.CreateFile({'id': file_id})
print('Downloaded content "{}"'.format(downloaded.GetContentString()))

ApiRequestError: <HttpError 404 when requesting https://www.googleapis.com/drive/v2/files/REPLACE_WITH_YOUR_FILE_ID?supportsAllDrives=true&alt=json returned "File not found: REPLACE_WITH_YOUR_FILE_ID". Details: "[{'message': 'File not found: REPLACE_WITH_YOUR_FILE_ID', 'domain': 'global', 'reason': 'notFound', 'location': 'file', 'locationType': 'other'}]">

The model names give you a hint about their capabilities and intended use:

Pro: These are the most capable models, ideal for complex reasoning, creative tasks, and detailed analysis.

Flash: These models are optimized for high speed and efficiency, making them great for summarization, chat applications, and tasks requiring rapid responses.

Gemma: These are lightweight, open-weight models suitable for a variety of text generation tasks and are great for experimentation.

In [ ]:
# Fetch a single <1MB file using the raw GitHub URL.
!curl --remote-name \
     -H 'Accept: application/vnd.github.v3.raw' \
     --location https://api.github.com/repos/jakevdp/PythonDataScienceHandbook/contents/notebooks/data/california_cities.csv

After executing the cell above, a new file named 'Sample file.txt' will appear in your [drive.google.com](https://drive.google.com/) file list.

In [ ]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np

# Randomly generated dataset of parking violations-
# Define the number of rows
num_rows = 1000000

states = ["NY", "NJ", "CA", "TX"]
violations = ["Double Parking", "Expired Meter", "No Parking",
              "Fire Hydrant", "Bus Stop"]
vehicle_types = ["SUBN", "SDN"]

# Create a date range
start_date = "2022-01-01"
end_date = "2022-12-31"
dates = pd.date_range(start=start_date, end=end_date, freq='D')

# Generate random data
data = {
    "Registration State": np.random.choice(states, size=num_rows),
    "Violation Description": np.random.choice(violations, size=num_rows),
    "Vehicle Body Type": np.random.choice(vehicle_types, size=num_rows),
    "Issue Date": np.random.choice(dates, size=num_rows),
    "Ticket Number": np.random.randint(1000000000, 9999999999, size=num_rows)
}

# Create a DataFrame
df = pd.DataFrame(data)

# Which parking violation is most commonly committed by vehicles from various U.S states?

(df[["Registration State", "Violation Description"]]  # get only these two columns
 .value_counts()  # get the count of offences per state and per type of offence
 .groupby("Registration State")  # group by state
 .head(1)  # get the first row in each group (the type of offence with the largest count)
 .sort_index()  # sort by state name
 .reset_index()
)

In [ ]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np

# Randomly generated dataset of parking violations-
# Define the number of rows
num_rows = 1000000

states = ["NY", "NJ", "CA", "TX"]
violations = ["Double Parking", "Expired Meter", "No Parking",
              "Fire Hydrant", "Bus Stop"]
vehicle_types = ["SUBN", "SDN"]

# Create a date range
start_date = "2022-01-01"
end_date = "2022-12-31"
dates = pd.date_range(start=start_date, end=end_date, freq='D')

# Generate random data
data = {
    "Registration State": np.random.choice(states, size=num_rows),
    "Violation Description": np.random.choice(violations, size=num_rows),
    "Vehicle Body Type": np.random.choice(vehicle_types, size=num_rows),
    "Issue Date": np.random.choice(dates, size=num_rows),
    "Ticket Number": np.random.randint(1000000000, 9999999999, size=num_rows)
}

# Create a DataFrame
df = pd.DataFrame(data)

# Which parking violation is most commonly committed by vehicles from various U.S states?

(df[["Registration State", "Violation Description"]]  # get only these two columns
 .value_counts()  # get the count of offences per state and per type of offence
 .groupby("Registration State")  # group by state
 .head(1)  # get the first row in each group (the type of offence with the largest count)
 .sort_index()  # sort by state name
 .reset_index()
)

# New Section

In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
from google.colab import drive
import shutil

drive.mount("/content/drive")

src = Path("/content/latif-brain/artifacts/lora")
dst = Path("/content/drive/MyDrive/LATIF-NI/models/lora-v1")

assert (src / "adapter_model.safetensors").is_file(), \
    f"Adapter missing: {src}"

    dst.mkdir(parents=True, exist_ok=True)

    shutil.copytree(src, dst, dirs_exist_ok=True)

    print("BACKUP SUCCESSFUL:", dst)

    for p in dst.rglob("*"):
        if p.is_file():
                print(p.name, p.stat().st_size, "bytes")

In [ ]:
from pathlib import Path
from google.colab import drive
import shutil

try:
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount status:", e)

src = Path("/content/latif-brain/artifacts/lora")
dst = Path("/content/drive/MyDrive/LATIF-NI/models/lora-v1")

assert (src / "adapter_model.safetensors").is_file(), "Training adapter not found in artifacts/lora"

dst.mkdir(parents=True, exist_ok=True)
shutil.copytree(src, dst, dirs_exist_ok=True)

print("Saved trained adapter to Google Drive:", dst)
print("Backing up evaluation report...")
eval_src = Path("/content/latif-brain/artifacts/evaluation.json")
if eval_src.is_file():
    shutil.copy(eval_src, dst / "evaluation.json")
    print("Saved evaluation.json to Google Drive")

print("Backed up files:")
for p in dst.rglob("*"):
    if p.is_file():
        print(f" - {p.name} ({p.stat().st_size} bytes)")

In [ ]:
import sys
# Install Gradio and Peft if not installed
!pip install -q gradio peft transformers accelerate bitsandbytes

import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
adapter_id = "/content/latif-brain/artifacts/lora"

print("Loading base model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Loading and applying fine-tuned LoRA adapter...")
model = PeftModel.from_pretrained(model, adapter_id)
model.eval()

def predict(message, history):
    # Prepare conversational system prompt or format
    messages = []

    # Robust parsing of Gradio's chat history across different versions
    for interaction in history:
        if isinstance(interaction, dict):
            # Newer Gradio versions pass history as dicts with role/content keys
            role = interaction.get("role", "")
            content = interaction.get("content", "")
            if role and content:
                messages.append({"role": role, "content": content})
        elif isinstance(interaction, (list, tuple)) and len(interaction) == 2:
            # Older Gradio versions pass history as tuples of (user_msg, assistant_msg)
            user_msg, assistant_msg = interaction
            if user_msg:
                messages.append({"role": "user", "content": user_msg})
            if assistant_msg:
                messages.append({"role": "assistant", "content": assistant_msg})

    messages.append({"role": "user", "content": message})

    # Generate tokenized prompt template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

    # Slice out the input prompt to get only the answer
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

# Create the Chat UI
demo = gr.ChatInterface(
    fn=predict,
    title="LATIF-NI Trained Model Dashboard",
    description="Test your fine-tuned model live in the browser using this chat companion UI."
)

print("Starting dashboard UI... Get ready to open the generated link below!")
demo.launch(share=True, debug=True)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from pathlib import Path

base_model_path = "Qwen/Qwen2.5-0.5B-Instruct"
adapter_path = "/content/latif-brain/artifacts/lora"
output_merged_dir = "/content/latif-qwen-merged"

print("Loading base model and adapter to merge them for mobile...")
tokenizer = AutoTokenizer.from_pretrained(base_model_path)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.float16,
    device_map="cpu"
)

# Load LoRA weights and merge them directly into the base architecture
model = PeftModel.from_pretrained(base_model, adapter_path)
merged_model = model.merge_and_unload()

# Save the model as a single, unified folder
merged_model.save_pretrained(output_merged_dir)
tokenizer.save_pretrained(output_merged_dir)

print(f"Unified model weights saved to: {output_merged_dir}")
print("This directory can now be compiled to GGUF using llama.cpp to be run offline on Android!")

In [ ]:
import zipfile
from pathlib import Path
import shutil
from google.colab import drive

# Mount Google Drive
try:
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount status:", e)

src_dir = Path("/content/latif-qwen-merged")
zip_path = Path("/content/LATIF-NI-UNIFIED-QWEN.zip")
drive_dest = Path("/content/drive/MyDrive/LATIF-NI")

print("Archiving unified model folder...")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in src_dir.rglob('*'):
        if p.is_file():
            z.write(p, p.relative_to(src_dir.parent).as_posix())

print(f"Unified archive created at {zip_path} ({zip_path.stat().st_size} bytes).")

if drive_dest.is_dir() or Path("/content/drive/MyDrive").is_dir():
    drive_dest.mkdir(parents=True, exist_ok=True)
    shutil.copy2(zip_path, drive_dest / zip_path.name)
    print(f"Successfully saved offline package to Google Drive at: {drive_dest / zip_path.name}")
else:
    print("Google Drive destination directory not found. Please verify mount.")

In [ ]:
# Re-run GGUF conversion with direct explicitly-checked pathing
import os
from pathlib import Path

# Force dependency install and check
!pip install -q gguf transformers HF-transfer

print("Converting HuggingFace merged weights to GGUF (f16)...")
!python /content/llama-cpp/convert_hf_to_gguf.py /content/latif-qwen-merged --outfile /content/latif-qwen-f16.gguf

f16_path = Path("/content/latif-qwen-f16.gguf")
if not f16_path.is_file():
    # Fallback search if convert_hf_to_gguf placed it inside the source folder
    found = list(Path("/content/latif-qwen-merged").glob("*.gguf"))
    if found:
        print(f"Found GGUF at alternative location: {found[0]}")
        found[0].rename(f16_path)

if f16_path.is_file():
    print(f"Success! f16 GGUF exists ({f16_path.stat().st_size} bytes). Quantizing to Q4_K_M...")
    # Perform quantization
    !/content/llama-cpp/build/bin/llama-quantize /content/latif-qwen-f16.gguf /content/latif-qwen-q4_k_m.gguf Q4_K_M

    # Save to Google Drive
    import shutil
    gguf_src = Path("/content/latif-qwen-q4_k_m.gguf")
    drive_dest = Path("/content/drive/MyDrive/LATIF-NI/latif-qwen-q4_k_m.gguf")
    if gguf_src.is_file():
        shutil.copy2(gguf_src, drive_dest)
        print(f"Success! Your optimized GGUF model has been saved to Google Drive: {drive_dest}")
else:
    print("Error: GGUF conversion failed. Please review standard output error messages above.")

In [ ]:
import subprocess
from pathlib import Path

model = 'Qwen/Qwen2.5-0.5B-Instruct'

print("Starting real GPU training run and capturing output...")
result = subprocess.run([
    'python', 'training/train_lora.py',
    '--model', model,
    '--data', 'training/data/train.jsonl',
    '--output', 'artifacts/lora',
    '--steps', '100',
    '--max-length', '256'
], capture_output=True, text=True)

print("--- STDOUT ---")
print(result.stdout)
print("--- STDERR ---")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"Training failed with exit code {result.returncode}")

adapter_path = Path('artifacts/lora/adapter_model.safetensors')
assert adapter_path.is_file(), 'Adapter file missing: Training failed to save weights.'
print('Training completed successfully! Adapter weight file exists at:', adapter_path)

In [ ]:
import subprocess
from pathlib import Path

model = 'Qwen/Qwen2.5-0.5B-Instruct'

print("Evaluating the trained model on held-out validation set...")
result = subprocess.run([
    'python', 'training/evaluate_lora.py',
    '--model', model,
    '--adapter', 'artifacts/lora',
    '--data', 'training/data/validation.jsonl'
], capture_output=True, text=True)

print("--- EVALUATION OUTPUT ---")
print(result.stdout)
print(result.stderr)

eval_json = Path('artifacts/evaluation.json')
if eval_json.is_file():
    print("\nSuccessfully saved evaluation metrics to:", eval_json)
    print(eval_json.read_text())
else:
    raise RuntimeError("Evaluation failed: Output metrics file not found.")

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

archive = '/content/LATIF-NI-QWEN-LORA.zip'
print("Creating ZIP archive containing the LoRA adapter weights and evaluation files...")

with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    # Compress LoRA weights and configs
    lora_dir = Path('artifacts/lora')
    if lora_dir.is_dir():
        for p in lora_dir.rglob('*'):
            if p.is_file():
                z.write(p, p.relative_to(Path('artifacts').parent).as_posix())
                print(f"Added: {p.as_posix()}")

    # Compress evaluation report
    eval_file = Path('artifacts/evaluation.json')
    if eval_file.is_file():
        z.write(eval_file, eval_file.as_posix())
        print(f"Added: {eval_file.as_posix()}")

    # Compress dataset manifest
    manifest_file = Path('training/data/manifest.json')
    if manifest_file.is_file():
        z.write(manifest_file, manifest_file.as_posix())
        print(f"Added: {manifest_file.as_posix()}")

print("\nArchive ready. Initiating browser download...")
files.download(archive)
print('Download initiated! This package contains the adapter weights and evaluation json.')

In [ ]:
from pathlib import Path
from google.colab import drive
import shutil

# Ensure Google Drive is mounted
try:
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount status:", e)

src_zip = Path("/content/LATIF-NI-QWEN-LORA.zip")
dst_dir = Path("/content/drive/MyDrive/LATIF-NI")

if src_zip.is_file():
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst_zip = dst_dir / src_zip.name
    shutil.copy2(src_zip, dst_zip)
    print(f"Successfully backed up ZIP archive to Google Drive: {dst_zip} ({dst_zip.stat().st_size} bytes)")
else:
    print(f"Error: Source archive {src_zip} not found. Please verify that the archiving step completed successfully.")